# IEX vs Okta Checker
So sánh trạng thái agent trong Okta snapshot với lịch IEX (dữ liệu đã qua cleaner).

In [57]:
import pandas as pd
from datetime import datetime

In [58]:
# Đọc dữ liệu từ file đã clean (iex_cleaner xuất ra)
iex_df = pd.read_excel('iex-data-extracted.xlsx')
okta_df = pd.read_csv('okta.csv')

# Xóa cột "Available On" nếu tồn tại
if "Available On" in okta_df.columns:
    okta_df = okta_df.drop(columns=["Available On"])

iex_df.head()

,IEX Id,Name,Shift,Activity,Start time,End time
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Open Time,1:00 PM,2:35 PM
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Break,2:35 PM,2:50 PM
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Open Time,2:50 PM,5:30 PM
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Lunch,5:30 PM,6:30 PM
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Open Time,6:30 PM,8:35 PM


In [59]:
# Đổi tên cột Okta cho đồng bộ (nếu cần chỉnh lại tuỳ dataset)
okta_df = okta_df.rename(columns={
    'userName': 'Name',
    'status': 'Activity',
    'duration': 'Duration'
})
okta_df['CheckTime'] = datetime.now()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,"Bui, Ngoc Thuan Vy",00:02:32,AVAILABLECHAT,1.0,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
1,"Bui, The Anh",00:00:26,BREAK,NaN,theanh.bui1@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
2,"Chinh, Ngoc Thu",00:04:07,AVAILABLECHAT,2.0,ngocthu.chinh@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,nguyenthaonhi.tran@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
3,"Dang, Anh Trung",00:25:16,LUNCH,NaN,anhtrung.dang@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
4,"Dang, Phuong Tien",00:11:23,AVAILABLECHAT,1.0,phuongtien.dang1@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,hoangkhoi.nguyen@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397


In [60]:
# Chuyển tất cả giá trị "Open Time" trong cột Activity thành "AVAILABLECHAT"
iex_df["Activity"] = iex_df["Activity"].replace("Open Time", "AVAILABLECHAT")

# Chuyển Start/End về datetime
iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')
iex_df.head()

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_13760\2382146022.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_13760\2382146022.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')


,IEX Id,Name,Shift,Activity,Start time,End time
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,AVAILABLECHAT,2025-08-21 13:00:00,2025-08-21 14:35:00
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Break,2025-08-21 14:35:00,2025-08-21 14:50:00
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,AVAILABLECHAT,2025-08-21 14:50:00,2025-08-21 17:30:00
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,Lunch,2025-08-21 17:30:00,2025-08-21 18:30:00
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,AVAILABLECHAT,2025-08-21 18:30:00,2025-08-21 20:35:00


In [61]:
import re

def clean_name(name: str) -> str:
    if pd.isna(name):
        return name
    # Thay dấu phẩy bằng khoảng trắng
    name = name.replace(",", " ")
    # Thêm khoảng trắng trước chữ in hoa (trừ chữ cái đầu)
    name = re.sub(r'(?<!^)(?=[A-Z])', ' ', name)
    # Chuẩn hoá khoảng trắng thừa
    name = " ".join(name.split())
    return name.strip()

# Chuẩn hoá cho cả IEX và Okta
iex_df["Name"] = iex_df["Name"].astype(str).map(clean_name)
okta_df["Agent Name"] = okta_df["Agent Name"].astype(str).map(clean_name)
#iex_df.head()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,Bui Ngoc Thuan Vy,00:02:32,AVAILABLECHAT,1.0,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
1,Bui The Anh,00:00:26,BREAK,NaN,theanh.bui1@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
2,Chinh Ngoc Thu,00:04:07,AVAILABLECHAT,2.0,ngocthu.chinh@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,nguyenthaonhi.tran@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
3,Dang Anh Trung,00:25:16,LUNCH,NaN,anhtrung.dang@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397
4,Dang Phuong Tien,00:11:23,AVAILABLECHAT,1.0,phuongtien.dang1@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,hoangkhoi.nguyen@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-21 04:28:09.205397


In [62]:
# So sánh giữa Okta và IEX
results = []
now = datetime.now()

for _, row in okta_df.iterrows():
    name = row['Agent Name']
    activity_okta = row['State']
    
    # Tìm agent trong IEX
    iex_agent = iex_df[iex_df['Name'] == name]
    iex_now = iex_agent[(iex_agent['Start time'] <= now) & (iex_agent['End time'] >= now)]
    
    if not iex_now.empty:
        activity_iex = iex_now.iloc[0]['Activity']
        start_iex = iex_now.iloc[0]['Start time']
        end_iex = iex_now.iloc[0]['End time']
    else:
        activity_iex = 'N/A'
        start_iex = None
        end_iex = None
    
    results.append({
        'Agent': name,
        'Activity_Okta': activity_okta,
        'Activity_IEX': activity_iex,
        'Start_IEX': start_iex,
        'End_IEX': end_iex,
        'Match': activity_okta.lower() == activity_iex.lower()
    })

result_df = pd.DataFrame(results)
result_df

,Agent,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
0,Bui Ngoc Thuan Vy,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 02:40:00,2025-08-21 05:00:00,True
1,Bui The Anh,BREAK,AVAILABLECHAT,2025-08-21 03:00:00,2025-08-21 05:00:00,False
2,Chinh Ngoc Thu,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 02:55:00,2025-08-21 05:00:00,True
3,Dang Anh Trung,LUNCH,Break,2025-08-21 04:15:00,2025-08-21 04:30:00,False
4,Dang Phuong Tien,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 03:20:00,2025-08-21 04:45:00,True
5,Dinh Thi Ngoc Han,AVAILABLECHAT,N/A,NaT,NaT,False
6,Duong Cong Hoang,OUTBOUNDCALL,AVAILABLECHAT,2025-08-21 03:15:00,2025-08-21 05:00:00,False
7,Duong Thi Thuy Duong,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 04:10:00,2025-08-21 06:00:00,True
8,Ho Ky Duyen,AVAILABLECHAT,Break,2025-08-21 04:20:00,2025-08-21 04:35:00,False
9,Ho Ngoc Huyen Trang,AVAILABLECHAT,AVAILABLECHAT,2025-08-21 03:10:00,2025-08-21 06:00:00,True


In [63]:
# Xuất mismatch ra file Excel
mismatch_df = result_df[(result_df['Match'] == False) & (result_df["Activity_IEX"] != "N/A")]
mismatch_df.to_excel('iex_okta_mismatch.xlsx', index=False)
mismatch_df

,Agent,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
1,Bui The Anh,BREAK,AVAILABLECHAT,2025-08-21 03:00:00,2025-08-21 05:00:00,False
3,Dang Anh Trung,LUNCH,Break,2025-08-21 04:15:00,2025-08-21 04:30:00,False
6,Duong Cong Hoang,OUTBOUNDCALL,AVAILABLECHAT,2025-08-21 03:15:00,2025-08-21 05:00:00,False
8,Ho Ky Duyen,AVAILABLECHAT,Break,2025-08-21 04:20:00,2025-08-21 04:35:00,False
10,Hoang Dai Hai,AVAILABLECHAT,Break,2025-08-21 04:15:00,2025-08-21 04:30:00,False
14,Le Thi Hue Anh,BREAK,AVAILABLECHAT,2025-08-21 04:15:00,2025-08-21 06:00:00,False
16,Lu Phuc Thanh Tai,TRAINING,AVAILABLECHAT,2025-08-21 04:00:00,2025-08-21 06:00:00,False
18,Nguyen Thi Kim Ngan,LOGIN,AVAILABLECHAT,2025-08-21 02:25:00,2025-08-21 05:00:00,False
20,Nguyen Dinh Tuan,LUNCH,AVAILABLECHAT,2025-08-21 03:35:00,2025-08-21 05:00:00,False
21,Nguyen Ha Tuan Thien,TRAINING,AVAILABLECHAT,2025-08-21 03:55:00,2025-08-21 05:00:00,False


In [64]:
# --- Export full comparison with both True/False ---
try:
    out_file = 'iex_okta_comparison.xlsx'
    result_df.to_excel(out_file, index=False)
    print(f'Saved full comparison to: {out_file}')
    result_df
except NameError as e:
    print('result_df is not defined. Please run the comparison cells above first.')
    raise


Saved full comparison to: iex_okta_comparison.xlsx
